In [1]:
config_path = "config/params.yaml"
mode = "default"

In [2]:
# Parameters
config_path = "config/params.yaml"


# 16 — Luaran paper

**Tujuan:** memeriksa kelengkapan figur dan tabel, membangun `macros.tex`, dan memastikan luaran bebas login serta nama repo.

In [3]:
import json
import re
from pathlib import Path

import pandas as pd

from ghremoval.config import load_context
from ghremoval.quality import Checks, pii_columns
from ghremoval.report import write_macros, write_numbers
from ghremoval.runlog import RunLog

ctx = load_context(config_path)
run = RunLog("16_paper_outputs", ctx.paths.run_log, ctx.paths.root)
checks = Checks()

## 1. Kelengkapan

In [4]:
listing = (ctx.paths.root / "docs/08-figures-and-tables.md").read_text(encoding="utf8")
names = sorted(set(re.findall(r"`((?:tab|fig)_[a-z0-9_]+)`", listing)))
existing = {p.stem for folder in (ctx.paths.tables, ctx.paths.figures) for p in folder.iterdir()}
missing = [n for n in names if n not in existing]
print("luaran terdaftar:", len(names), "tidak ada:", missing)

luaran terdaftar: 27 tidak ada: ['tab_audit_results']


## 2. Angka dan makro

In [5]:
write_numbers({"snapshot_date": ctx.p.label["snapshot_date"]})
numbers = json.loads(ctx.paths.numbers.read_text())
write_macros(numbers, ctx.paths.results / "macros.tex")
print(json.dumps(numbers, indent=1)[:1500])

{
 "aurt_delta_h4a": 0.0582010582010582,
 "delta_ap_h1a": 0.17346627039468782,
 "delta_ap_h5": 0.10728314756761514,
 "n_accounts_study": 847681,
 "n_events_study": 5019698,
 "n_fixed_cohort": 159209,
 "positives_f3_5_19": 471,
 "primary_id_block_width": 20000,
 "ratio_h2a": 1.1759642207634509,
 "snapshot_date": "2026-11-10",
 "within_cell_auc_h1b": 0.7355678510858691
}


## 3. Cek privasi pada tabel

In [6]:
bad = {}
for path in ctx.paths.tables.glob("*.csv"):
    columns = list(pd.read_csv(path, nrows=1).columns)
    found = pii_columns(columns)
    if found:
        bad[path.name] = found
checks.add("Q20", not bad, f"kolom mencurigakan: {bad}")
checks.add("kelengkapan_luaran", not missing or set(missing) <= {"tab_audit_results"}, f"tidak ada: {missing}")
checks.raise_if_failed()
print(run.finish(n_outputs=len(existing)))

[LULUS] Q20: kolom mencurigakan: {}
[LULUS] kelengkapan_luaran: tidak ada: ['tab_audit_results']
{'notebook': '16_paper_outputs', 'finished_at_utc': '2026-11-10T12:22:41+00:00', 'git_commit': 'dde2836', 'duration_seconds': 0.1, 'counts': {'n_outputs': 36}}
